In [14]:
"""
Hybrid RAG Pipeline — Dense (Qwen3-Embedding) + Sparse (BM25) with Qdrant
==========================================================================
Combines semantic similarity (dense vectors) with keyword matching (sparse vectors)
using Reciprocal Rank Fusion (RRF) for final scoring.
"""
 
import os
from pathlib import Path
 
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    NamedSparseVector,
    PointStruct,
    SparseIndexParams,
    SparseVector,
    SparseVectorParams,
    VectorParams,
)

In [15]:
from langchain_docling.loader import DoclingLoader, ExportType
from docling.chunking import HybridChunker
 
# ── Config ──────────────────────────────────────────────────────────────
DENSE_MODEL = "Qwen/Qwen3-Embedding-0.6B"
COLLECTION = "hybrid_test"
QDRANT_URL = "http://localhost:6333"
FILE_PATH = "./../data/Manuals/mds_axis_compensation_en.pdf"
CHUNK_MAX_TOKENS = 300
 
# ── 1. Init embeddings ─────────────────────────────────────────────────
print("[1/6] Loading dense embedding model...")
dense_embeddings = HuggingFaceEmbeddings(model_name=DENSE_MODEL)
dense_size = len(dense_embeddings.embed_query("test"))
print(f"       Dense vector dim: {dense_size}")

[1/6] Loading dense embedding model...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5868.46it/s]


       Dense vector dim: 1024


In [16]:
# ── 2. Sparse (BM25-like) encoder ──────────────────────────────────────
# Lightweight token-frequency sparse vectors — no extra model needed.
import re
import math
from collections import Counter
from typing import Optional
 
 
class BM25SparseEncoder:
    """
    Builds sparse vectors from token frequencies using BM25-style TF weighting.
    Vocabulary is built incrementally during indexing, then reused at query time.
    """
 
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.vocab: dict[str, int] = {}  # token -> integer index
        self.doc_freq: Counter = Counter()  # token -> nb docs containing it
        self.doc_count: int = 0
        self.avg_dl: float = 0.0
        self._total_tokens: int = 0
 
    @staticmethod
    def _tokenize(text: str) -> list[str]:
        return re.findall(r"[a-zA-Z0-9\-]+", text.lower())
 
    def fit(self, documents: list[str]) -> "BM25SparseEncoder":
        """Build vocabulary + IDF stats from a corpus."""
        for doc in documents:
            tokens = self._tokenize(doc)
            self.doc_count += 1
            self._total_tokens += len(tokens)
            unique = set(tokens)
            for t in unique:
                self.doc_freq[t] += 1
            for t in tokens:
                if t not in self.vocab:
                    self.vocab[t] = len(self.vocab)
        self.avg_dl = self._total_tokens / max(self.doc_count, 1)
        return self
 
    def encode(self, text: str) -> SparseVector:
        """Encode a single text into a Qdrant SparseVector."""
        tokens = self._tokenize(text)
        tf = Counter(tokens)
        dl = len(tokens)
 
        indices: list[int] = []
        values: list[float] = []
 
        for token, freq in tf.items():
            if token not in self.vocab:
                # Unknown token at query time — skip
                continue
            idx = self.vocab[token]
            # BM25 TF component
            tf_score = (freq * (self.k1 + 1)) / (
                freq + self.k1 * (1 - self.b + self.b * dl / max(self.avg_dl, 1))
            )
            # IDF component
            df = self.doc_freq.get(token, 0)
            idf = math.log(1 + (self.doc_count - df + 0.5) / (df + 0.5))
            score = tf_score * idf
            if score > 0:
                indices.append(idx)
                values.append(round(score, 4))
 
        return SparseVector(indices=indices, values=values)
 
 
print("[2/6] Sparse BM25 encoder ready.")

[2/6] Sparse BM25 encoder ready.


In [17]:
# ── 3. Document loading + chunking ─────────────────────────────────────
print(f"[3/6] Loading & chunking: {FILE_PATH}")
loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=ExportType.DOC_CHUNKS,
    chunker=HybridChunker(
        tokenizer=DENSE_MODEL,
        max_tokens=CHUNK_MAX_TOKENS,
        merge_peers=True,
        repeat_table_header=True,
        omit_header_on_overflow=True,
    ),
)
docs = loader.load()
texts = [doc.page_content for doc in docs]
metadatas = [doc.metadata for doc in docs]
print(f"       {len(docs)} chunks produced.")

[3/6] Loading & chunking: ./../data/Manuals/mds_axis_compensation_en.pdf


[INFO] 2026-04-10 17:37:30,131 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-10 17:37:30,133 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-10 17:37:30,149 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-10 17:37:30,150 [RapidOCR] main.py:50: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-10 17:37:30,340 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-10 17:37:30,341 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-10 17:37:30,343 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-04-10 17:37:30,344 [RapidOCR] main.py:50: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\

       129 chunks produced.


In [19]:
# ── 4. Fit sparse encoder + create Qdrant collection ───────────────────
print("[4/6] Fitting BM25 on corpus & creating collection...")
sparse_encoder = BM25SparseEncoder().fit(texts)
 
client = QdrantClient(url=QDRANT_URL, grpc_port=6334, prefer_grpc=True)
 
# Recreate collection with both dense + sparse vector configs
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
 
client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": VectorParams(size=dense_size, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(index=SparseIndexParams()),
    },
)

[4/6] Fitting BM25 on corpus & creating collection...


True

In [20]:
# ── 5. Index documents ─────────────────────────────────────────────────
print("[5/6] Encoding & indexing chunks...")
BATCH_SIZE = 32
 
for batch_start in range(0, len(texts), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(texts))
    batch_texts = texts[batch_start:batch_end]
    batch_metas = metadatas[batch_start:batch_end]
 
    # Dense embeddings (batched)
    dense_vecs = dense_embeddings.embed_documents(batch_texts)
    # Sparse vectors
    sparse_vecs = [sparse_encoder.encode(t) for t in batch_texts]
 
    points = [
        PointStruct(
            id=batch_start + i,
            vector={
                "dense": dense_vecs[i],
                "sparse": sparse_vecs[i],
            },
            payload={
                "text": batch_texts[i],
                "metadata": batch_metas[i],
            },
        )
        for i in range(len(batch_texts))
    ]
    client.upsert(collection_name=COLLECTION, points=points)
 
print(f"       {len(texts)} chunks indexed (dense + sparse).")

[5/6] Encoding & indexing chunks...
       129 chunks indexed (dense + sparse).


In [23]:
# ── 6. Hybrid search with RRF ──────────────────────────────────────────
def hybrid_search(
    query: str,
    k: int = 15,
    dense_weight: float = 0.6,
    sparse_weight: float = 0.4,
    prefetch_k: int = 40,
) -> list[dict]:
    """
    Run dense + sparse searches in parallel, fuse with Reciprocal Rank Fusion.
 
    Args:
        query:          Natural language query
        k:              Final number of results
        dense_weight:   Weight for dense (semantic) scores in RRF
        sparse_weight:  Weight for sparse (keyword) scores in RRF
        prefetch_k:     Number of candidates per search leg
    """
    # Encode query
    dense_vec = dense_embeddings.embed_query(query)
    sparse_vec = sparse_encoder.encode(query)
 
    # query_points — new qdrant-client API (v1.12+)
    dense_response = client.query_points(
        collection_name=COLLECTION,
        query=dense_vec,
        using="dense",
        limit=prefetch_k,
        with_payload=True,
    )
    sparse_response = client.query_points(
        collection_name=COLLECTION,
        query=sparse_vec,
        using="sparse",
        limit=prefetch_k,
        with_payload=True,
    )
 
    dense_results = dense_response.points
    sparse_results = sparse_response.points
 
    # ─── Reciprocal Rank Fusion ───
    RRF_K = 60  # smoothing constant
    scores: dict[int, float] = {}
    payloads: dict[int, dict] = {}
 
    for rank, hit in enumerate(dense_results):
        pid = hit.id
        scores[pid] = scores.get(pid, 0) + dense_weight / (RRF_K + rank + 1)
        payloads[pid] = hit.payload
 
    for rank, hit in enumerate(sparse_results):
        pid = hit.id
        scores[pid] = scores.get(pid, 0) + sparse_weight / (RRF_K + rank + 1)
        payloads[pid] = hit.payload
 
    # Sort by fused score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
 
    return [
        {
            "id": pid,
            "score": round(score, 6),
            "text": payloads[pid]["text"],
            "metadata": payloads[pid].get("metadata", {}),
        }
        for pid, score in ranked
    ]

In [24]:
# ── Run query ───────────────────────────────────────────────────────────
print("\n[6/6] Running hybrid search...")
query = "give me the structure, parameter et functionality for P-COMP-00047"
results = hybrid_search(query, k=15, dense_weight=0.6, sparse_weight=0.4)
 
for i, r in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"RESULT {i}  |  score={r['score']}  |  id={r['id']}")
    print(f"{'='*60}")
    print(r["text"][:500])
    print(f"\nMetadata: {r['metadata']}")
 



[6/6] Running hybrid search...

RESULT 1  |  score=0.01555  |  id=126
P
 P-COMP-00031....................................................., 1 = 20. P-COMP-00032....................................................., 1 = . P-COMP-00033....................................................., 1 = 21. P-COMP-00041....................................................., 1 = 34. P-COMP-00042....................................................., 1 = 34. P-COMP-00043....................................................., 1 = 35. P-COMP-00044.....................................

Metadata: {'dl_meta': {'doc_items': [{'prov': [{'charspan': [0, 0], 'bbox': {'b': 241.2552490234375, 'l': 55.77126693725586, 'coord_origin': 'BOTTOMLEFT', 't': 725.1133728027344, 'r': 290.9179382324219}, 'page_no': 37}], 'children': [], 'self_ref': '#/tables/54', 'label': 'document_index', 'parent': {'$ref': '#/body'}, 'content_layer': 'body'}], 'origin': {'binary_hash': 5388167723037981658, 'filename': 'mds_axis_compensati

In [ ]:
# ── Comparison helper (optional) ────────────────────────────────────────
def compare_modes(query: str, k: int = 5):
    """Side-by-side: dense only vs sparse only vs hybrid."""
    print(f"\n{'─'*70}")
    print(f"QUERY: {query}")
    print(f"{'─'*70}")
 
    dense_only = hybrid_search(query, k=k, dense_weight=1.0, sparse_weight=0.0)
    sparse_only = hybrid_search(query, k=k, dense_weight=0.0, sparse_weight=1.0)
    hybrid = hybrid_search(query, k=k, dense_weight=0.6, sparse_weight=0.4)
 
    for label, res in [("DENSE", dense_only), ("SPARSE", sparse_only), ("HYBRID", hybrid)]:
        print(f"\n  [{label}]")
        for i, r in enumerate(res, 1):
            preview = r["text"][:80].replace("\n", " ")
            print(f"    {i}. (id={r['id']}, score={r['score']}) {preview}...")
 
    print()